# FLAME 2 segmentation labels: a look at what the export writes

FLAME 2 only says whether a frame has fire, so the fire masks come from the thermal frames. A fixed grayscale threshold does not work: the thermal frames are false-colour JPEGs, the palette changes at about frame 17,370, and the colours are rescaled every frame.

The labelling logic lives in `src/embereye/data/flame2.py` (palette detection, `hot_mask`, `mask_to_yolo_seg`), and the full export is a script:

```bash
python scripts/export_flame2_yolo.py    # block split + YOLO-seg labels -> data/flame2/yolo_seg.nosync/
```

This notebook only looks at single frames. How well the masks match hand labels: `docs/findings/flame2-label-validation.md`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from embereye.data import flame2 as f2
from embereye.masks import polygons_to_mask

labels = f2.parse_labels(f2.FLAME2_DIR / "labels.txt")
GATE = 0.25    # the frame gate used for the training labels

## Frames from both palettes, and a no-fire frame

The label each frame gets in the export: no-fire frames are always empty (labels.txt is the truth); fire frames get `hot_mask` with the gate on, written as YOLO-seg outlines.

In [ ]:
def show(frame_num: int) -> None:
    rgb, thermal = f2.load_frame(frame_num)
    fire, smoke = labels[frame_num]
    variant = f2.palette_variant(thermal)
    if fire and variant != f2.UNKNOWN:
        palette = f2.INFERNO if variant == f2.INFERNO else f2.RAINBOW
        mask = f2.hot_mask(thermal, palette, max_frame_median=GATE)
    else:
        mask = np.zeros(thermal.shape[:2], dtype=bool)
    lines = f2.mask_to_yolo_seg(mask)
    outlines = [np.array(line.split()[1:], float).reshape(-1, 2) * [254, 254] for line in lines]
    label_mask = polygons_to_mask(outlines, mask.shape)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(rgb[..., ::-1]); axes[0].set_title("RGB")
    axes[1].imshow(thermal[..., ::-1]); axes[1].set_title(f"thermal ({variant})")
    axes[2].imshow(thermal[..., ::-1]); axes[2].contour(label_mask, colors="cyan", linewidths=0.8)
    axes[2].set_title(f"label: {len(lines)} region(s)")
    for ax in axes:
        ax.axis("off")
    flag = ("YY" if smoke else "YN") if fire else "no fire"
    fig.suptitle(f"frame {frame_num}: {flag}")
    plt.show()


for frame_num in (5000, 15000, 30000, 46482):    # no-fire, inferno, rainbow-red, rainbow-magenta
    show(frame_num)

## The export report

After running the export, its report lists the blocks, the frames per split and stratum, the checks and the caveats.

In [ ]:
report = f2.FLAME2_DIR / "yolo_seg.nosync" / "split_report.md"
print(report.read_text() if report.exists() else "Run: python scripts/export_flame2_yolo.py")